In [9]:
import numpy as np
import pandas as pd
import os
import librosa
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Input
from tensorflow.keras.layers import Conv2D, MaxPooling2D, LSTM, TimeDistributed
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tqdm import tqdm

In [10]:
# Load metadata and prepare paths

audio_dataset_path = 'E:\\Users\\Sumit\\Downloads\\UrbanSound8K\\UrbanSound8K\\audio\\'
metadata = pd.read_csv('E:\\Users\\Sumit\\Downloads\\UrbanSound8K\\UrbanSound8K\\metadata\\UrbanSound8K.csv')

In [11]:
def features_extractor(file_name):
    audio, sample_rate = librosa.load(file_name, res_type='kaiser_fast')
    mfccs_features = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
    mfccs_scaled_features = np.mean(mfccs_features.T, axis=0)
    return mfccs_scaled_features

In [12]:
# Prepare the dataset (already done in the previous ANN code)
extracted_features = []
for index_num, row in tqdm(metadata.iterrows()):
    file_name = os.path.join(os.path.abspath(audio_dataset_path),'fold'+str(row["fold"])+'/',str(row["slice_file_name"]))
    final_class_labels = row["class"]
    data = features_extractor(file_name)
    extracted_features.append([data,final_class_labels])

3555it [07:11,  8.70it/s]C:\Users\sumit\anaconda3\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1323
  warnings.warn(
8325it [16:13, 13.47it/s]C:\Users\sumit\anaconda3\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1103
  warnings.warn(
8329it [16:13, 17.93it/s]C:\Users\sumit\anaconda3\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1523
  warnings.warn(
8732it [16:55,  8.60it/s]


In [13]:
extracted_features_df = pd.DataFrame(extracted_features, columns=['feature','class'])

In [14]:
# Convert extracted features to arrays
X = np.array(extracted_features_df['feature'].tolist())
y = np.array(extracted_features_df['class'].tolist())

# Encode labels
labelencoder = LabelEncoder()
y = to_categorical(labelencoder.fit_transform(y))

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
# Reshape for CNN-LSTM input: [samples, time_steps, features, channels]
X_train = X_train.reshape(X_train.shape[0], 1, X_train.shape[1], 1)  # (num_samples, time_steps=1, num_features, channels=1)
X_test = X_test.reshape(X_test.shape[0], 1, X_test.shape[1], 1)

y_train = y_train.reshape(y_train.shape[0], 1, y_train.shape[1])
y_test = y_test.reshape(y_test.shape[0], 1, y_test.shape[1])

In [16]:
# # CRNN Model Definition
# model = Sequential()

# # 1st Convolutional Layer
# model.add(TimeDistributed(Conv2D(64, (3, 3), activation='relu', padding='same'), input_shape=(None, 40, 40, 1)))
# model.add(TimeDistributed(MaxPooling2D(pool_size=(2, 2))))
# model.add(Dropout(0.3))

# # 2nd Convolutional Layer
# model.add(TimeDistributed(Conv2D(128, (3, 3), activation='relu', padding='same')))
# model.add(TimeDistributed(MaxPooling2D(pool_size=(2, 2))))
# model.add(Dropout(0.3))

# model.add(TimeDistributed(Flatten()))
# model.add(Dense(128, activation='relu'))
# model.add(Dropout(0.3))
# model.add(Dense(10, activation='softmax'))

# # Flatten the data
# model.add(TimeDistributed(Flatten()))

# # LSTM Layer (for temporal patterns)
# model.add(LSTM(128, return_sequences=False, dropout=0.5))

# # Fully connected layers
# model.add(Dense(100, activation='relu'))
# model.add(Dropout(0.3))

# # Output layer
# num_labels = y.shape[1]
# model.add(Dense(num_labels, activation='softmax'))

In [17]:
# Model Definition
model = Sequential()

# Adjust input shape to ensure enough spatial size (e.g., (None, 40, 40, 1))
# If your input is smaller, adjust the first Conv2D layer accordingly
model.add(TimeDistributed(Conv2D(64, (3, 3), activation='relu', padding='same'), input_shape=(None, 40, 1, 1)))
model.add(TimeDistributed(MaxPooling2D(pool_size=(1, 2), padding = 'same')))
model.add(Dropout(0.3))

# 2nd Convolutional Layer with padding
model.add(TimeDistributed(Conv2D(128, (3, 3), activation='relu', padding='same')))
model.add(TimeDistributed(MaxPooling2D(pool_size=(1, 2), padding = 'same')))
model.add(Dropout(0.3))

# 3rd Convolutional Layer (if needed)
model.add(TimeDistributed(Conv2D(256, (3, 3), activation='relu', padding='same')))
model.add(TimeDistributed(MaxPooling2D(pool_size=(1, 2), padding = 'same')))
model.add(Dropout(0.3))

# Flatten and Dense layers
model.add(TimeDistributed(Flatten()))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(10, activation='softmax'))  # Output layer

# Model Summary
model.summary()

C:\Users\sumit\anaconda3\Lib\site-packages\keras\src\layers\core\wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, None, 40, 1,    │           640 │
│ (TimeDistributed)               │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, None, 40, 1,    │             0 │
│ (TimeDistributed)               │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, None, 40, 1,    │             0 │
│                                 │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, None, 40, 1,    │        73,856 │
│ (TimeDistributed)               │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, None, 40, 1,    │             0 │
│ (TimeDistributed)               │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, None, 40, 1,    │             0 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ (None, None, 40, 1,    │       295,168 │
│ (TimeDistributed)               │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_5              │ (None, None, 40, 1,    │             0 │
│ (TimeDistributed)               │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, None, 40, 1,    │             0 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_6              │ (None, None, 10240)    │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 128)      │     1,310,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, None, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, None, 10)       │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,681,802 (6.42 MB)

 Trainable params: 1,681,802 (6.42 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])

# Model Summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, None, 40, 1,    │           640 │
│ (TimeDistributed)               │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, None, 40, 1,    │             0 │
│ (TimeDistributed)               │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, None, 40, 1,    │             0 │
│                                 │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, None, 40, 1,    │        73,856 │
│ (TimeDistributed)               │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, None, 40, 1,    │             0 │
│ (TimeDistributed)               │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, None, 40, 1,    │             0 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ (None, None, 40, 1,    │       295,168 │
│ (TimeDistributed)               │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_5              │ (None, None, 40, 1,    │             0 │
│ (TimeDistributed)               │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, None, 40, 1,    │             0 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_6              │ (None, None, 10240)    │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 128)      │     1,310,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, None, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, None, 10)       │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,681,802 (6.42 MB)

 Trainable params: 1,681,802 (6.42 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
from keras.utils import to_categorical

# Assuming you have 10 classes
#y_train = to_categorical(y_train, num_classes=10)
#y_test = to_categorical(y_test, num_classes=10)

In [20]:
from tensorflow.keras.callbacks import ModelCheckpoint
from datetime import datetime

# Training the model
num_epochs = 100
num_batch_size = 32

# checkpointer = tf.keras.callbacks.ModelCheckpoint(filepath='E:\\Users\\Sumit\Downloads\\UrbanSound8K\\UrbanSound8K\\saved_models_CRNN\\audio_classification_CRNN.keras', 
#                                                   verbose=1, save_best_only=True)

checkpointer = ModelCheckpoint(filepath='E:/Users/Sumit/Downloads/UrbanSound8K/UrbanSound8K/saved_models_CRNN/audio_classification_CRNN.keras',
                                                  verbose=1, save_best_only=True)

start = datetime.now()

# Train
#model.fit(X_train, y_train, batch_size=num_batch_size, epochs=num_epochs, validation_data=(X_test, y_test), callbacks=[checkpointer])
model.fit(X_train, y_train, batch_size=num_batch_size, epochs=num_epochs, validation_data=(X_test, y_test), callbacks=[checkpointer])

duration = datetime.now() - start
print("Training completed in time: ", duration)  

Epoch 1/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - accuracy: 0.3468 - loss: 1.8298
Epoch 1: val_loss improved from inf to 1.05055, saving model to E:/Users/Sumit/Downloads/UrbanSound8K/UrbanSound8K/saved_models_CRNN/audio_classification_CRNN.keras
219/219 ━━━━━━━━━━━━━━━━━━━━ 36s 136ms/step - accuracy: 0.3473 - loss: 1.8286 - val_accuracy: 0.6594 - val_loss: 1.0506
Epoch 2/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 0.6218 - loss: 1.1277
Epoch 2: val_loss improved from 1.05055 to 0.78532, saving model to E:/Users/Sumit/Downloads/UrbanSound8K/UrbanSound8K/saved_models_CRNN/audio_classification_CRNN.keras
219/219 ━━━━━━━━━━━━━━━━━━━━ 28s 129ms/step - accuracy: 0.6218 - loss: 1.1275 - val_accuracy: 0.7413 - val_loss: 0.7853
Epoch 3/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - accuracy: 0.7001 - loss: 0.8880
Epoch 3: val_loss improved from 0.78532 to 0.61942, saving model to E:/Users/Sumit/Downloads/UrbanSound8K/UrbanSound8K/saved_models_CRNN/audio_classification_C

In [24]:
# # Evaluation
# test_accuracy = model.evaluate(X_test, y_test, verbose=0)
# print("Test accuracy: {:.2f}%".format(test_accuracy[1] * 100))

# # Prediction for a sample file
# filename = 'E:\\Users\\Sumit\\Downloads\\UrbanSound8K\\UrbanSound8K\\gun-shot.mp3'
# prediction_feature = features_extractor(filename)
# prediction_feature = prediction_feature.reshape(1, 1, prediction_feature.shape[0], 1)

# predictions = model.predict(prediction_feature)
# predicted_class = np.argmax(predictions, axis=-1)
# prediction_label = labelencoder.inverse_transform(predicted_class)
# print(f"Predicted class: {prediction_label}")

Test accuracy: 91.53%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 361ms/step


ValueError: y should be a 1d array, got an array of shape (1, 10) instead.

In [36]:
# Prediction for a sample file
filename = 'E:\\Users\\Sumit\\Downloads\\UrbanSound8K\\UrbanSound8K\\siren.wav'
prediction_feature = features_extractor(filename)
prediction_feature = prediction_feature.reshape(1, 1, prediction_feature.shape[0], 1)

# Make predictions
predictions = model.predict(prediction_feature)

# Get predicted class
predicted_class = np.argmax(predictions, axis=-1).flatten()

# Inverse transform to get original class label
prediction_label = labelencoder.inverse_transform(predicted_class)

print(f"Predicted class: {prediction_label}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
Predicted class: ['siren']
